# 05_Tài – MULTIVARIATE EDA

## Mục tiêu
- Phân tích mối quan hệ giữa các biến trong `cleaned_data.csv`.
- Phân tích mối tương quan giữa các biến số bằng Pearson và Spearman.
- Phân tích quan hệ giữa biến số và biến phân loại.
- Phân tích mối quan hệ giữa các biến phân loại.
- Trực quan hóa bằng Heatmap, Scatter Plot, Grouped Bar Chart, Grouped Boxplot và Contingency Table/Heatmap.
- Mỗi biểu đồ có Title, X/Y, Legend phù hợp và phần **So What?**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("cleaned_data.csv")

print("Kích thước dữ liệu:", df.shape)
display(df.head())


## 1. Phân tích Numerical × Numerical

Chọn 3 biến số đại diện để tập trung phân tích:
- `price`
- `monthly_views`
- `monthly_orders`


In [ ]:
numeric_cols = ["price", "monthly_views", "monthly_orders"]

numeric_df = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

print("Ma trận tương quan Pearson:")
pearson_corr = numeric_df.corr(method="pearson")
display(pearson_corr)

print("Ma trận tương quan Spearman:")
spearman_corr = numeric_df.corr(method="spearman")
display(spearman_corr)


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    pearson_corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Heatmap tương quan Pearson giữa các biến số")
plt.xlabel("Biến số")
plt.ylabel("Biến số")
plt.tight_layout()
plt.show()

print("So What:")
print("Heatmap cho biết mức độ và chiều hướng tương quan giữa các biến số. "
      "Các cặp có tương quan cao có thể cần được xem xét khi lựa chọn biến đầu vào cho mô hình AI.")


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x="monthly_views",
    y="monthly_orders",
    alpha=0.6,
    label="Sản phẩm"
)
plt.title("Mối quan hệ giữa lượt xem và số đơn hàng")
plt.xlabel("Monthly Views")
plt.ylabel("Monthly Orders")
plt.legend()
plt.tight_layout()
plt.show()

print("So What:")
print("Biểu đồ giúp quan sát xu hướng giữa lượt xem và số đơn hàng. "
      "Nếu các điểm có xu hướng tăng cùng nhau, lượt xem có thể là biến hữu ích khi phân tích hoặc dự đoán số đơn hàng.")


## 2. Phân tích Categorical × Numerical

Sử dụng `source` làm biến phân loại và `price` làm biến số để so sánh giữa các nguồn dữ liệu.


In [ ]:
group_price = (
    df.groupby("source", dropna=False)["price"]
      .mean()
      .sort_values(ascending=False)
)

display(group_price.to_frame("mean_price"))


In [ ]:
plt.figure(figsize=(8, 5))
group_price.plot(kind="bar")
plt.title("Giá trung bình theo nguồn dữ liệu")
plt.xlabel("Source")
plt.ylabel("Mean Price")
plt.legend(["Mean Price"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("So What:")
print("Grouped Bar Chart cho phép so sánh giá trung bình giữa các nguồn dữ liệu. "
      "Sự khác biệt giữa các nhóm cần được xem xét khi kết hợp dữ liệu từ nhiều nguồn.")


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="source",
    y="price"
)
plt.title("Phân bố giá theo nguồn dữ liệu")
plt.xlabel("Source")
plt.ylabel("Price")
plt.legend([], [], frameon=False)
plt.tight_layout()
plt.show()

print("So What:")
print("Grouped Boxplot cho thấy trung vị, độ phân tán và các giá trị ngoại lệ của giá theo từng nguồn dữ liệu. "
      "Điều này giúp phát hiện sự khác biệt về phân phối giữa các nguồn.")


## 3. Phân tích Categorical × Categorical

Phân tích mối quan hệ giữa `source` và `tv_type` bằng bảng tần số chéo và biểu đồ.


In [ ]:
cross_tab = pd.crosstab(
    df["source"],
    df["tv_type"],
    margins=True
)

display(cross_tab)


In [ ]:
ct = pd.crosstab(df["source"], df["tv_type"])

plt.figure(figsize=(10, 6))
sns.heatmap(
    ct,
    annot=True,
    fmt="d",
    cmap="Blues"
)
plt.title("Heatmap số lượng sản phẩm theo Source và TV Type")
plt.xlabel("TV Type")
plt.ylabel("Source")
plt.tight_layout()
plt.show()

print("So What:")
print("Heatmap cho thấy số lượng sản phẩm của từng loại TV trong mỗi nguồn dữ liệu. "
      "Sự phân bố khác nhau giữa các nhóm có thể ảnh hưởng đến cách cân bằng và sử dụng dữ liệu cho mô hình AI.")


In [ ]:
# Chọn các nhóm TV có nhiều dữ liệu để biểu đồ dễ quan sát
top_tv_types = df["tv_type"].value_counts().head(5).index
bar_df = (
    df[df["tv_type"].isin(top_tv_types)]
    .groupby(["source", "tv_type"])
    .size()
    .unstack(fill_value=0)
)

bar_df.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Số lượng sản phẩm theo Source và TV Type")
plt.xlabel("Source")
plt.ylabel("Number of Products")
plt.legend(title="TV Type")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("So What:")
print("Grouped Bar Chart giúp so sánh số lượng từng loại TV giữa các nguồn dữ liệu. "
      "Biểu đồ hỗ trợ kiểm tra sự khác biệt về cơ cấu dữ liệu trước khi xây dựng mô hình.")


## 4. Kết luận Multivariate EDA

Các phân tích trên giúp:
- Xác định mối tương quan giữa các biến số.
- Quan sát xu hướng giữa các biến số.
- So sánh biến số giữa các nhóm phân loại.
- Kiểm tra sự phân bố giữa các nhóm phân loại.
- Phát hiện các mối quan hệ có thể hữu ích cho bước lựa chọn đặc trưng và xây dựng mô hình AI.
